In [1]:
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler

from CustomSpeachDataset import CustomSpeachDataset
from SpeachClassifierModel import SpeachClassifierModel

In [2]:
dataset = CustomSpeachDataset(Path("preprocessed_dataset"), preload=True)

Preloading dataset from disk...


/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_indices, test_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.y,
    random_state=42
)

class_weights = 1.0 / dataset.label_counts
all_sample_weights = np.array([class_weights[y] for y in dataset.y])

train_dataset = torch.utils.data.Subset(dataset, train_indices)
train_sample_weights = all_sample_weights[train_indices]
len(train_sample_weights)

16447

In [4]:
def train_new_model(epochs: int, dataloader: DataLoader) -> SpeachClassifierModel:
    model = SpeachClassifierModel()

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()

        for data, target in dataloader:
            y_pred = model(data)

            loss = loss_fn(y_pred, target)
            loss.backward()

            optimizer.step()

    return model

In [5]:
sampler = WeightedRandomSampler(
    weights=train_sample_weights,
    num_samples=len(train_sample_weights),
    replacement=True,
)
loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)

train_new_model(1, loader)

SpeachClassifierModel(
  (model): Sequential(
    (0): Conv1d(1, 8, kernel_size=(13,), stride=(1,))
    (1): MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
    (2): Dropout(p=0.3, inplace=False)
    (3): Conv1d(8, 16, kernel_size=(11,), stride=(1,))
    (4): MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
    (5): Dropout(p=0.3, inplace=False)
    (6): Conv1d(16, 32, kernel_size=(9,), stride=(1,))
    (7): MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
    (8): Dropout(p=0.3, inplace=False)
    (9): Conv1d(32, 64, kernel_size=(7,), stride=(1,))
    (10): MaxPool1d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
    (11): Dropout(p=0.3, inplace=False)
    (12): Flatten(start_dim=1, end_dim=-1)
    (13): Linear(in_features=6080, out_features=256, bias=True)
    (14): Dropout(p=0.3, inplace=False)
    (15): Linear(in_features=256, out_features=128, bias=True)
    (16): Dropout(p=0.3, inplac